파이썬의 **`ast` (Abstract Syntax Tree, 추상 구문 트리)** 모듈은 파이썬 소스 코드를 문자열 형태 그대로 실행하는 대신, **문법 구조를 소나무/나뭇가지 형태의 객체 트리로 변환하여 분석, 검증, 변환**할 수 있게 해주는 파이썬 표준 라이브러리입니다.

---

### 1. `ast` 모듈을 사용하는 이유 (핵심 목적)

1. **보안 및 샌드박싱 (Sandboxing)**
* `eval()`이나 `exec()`는 외부 사용자의 악의적인 파이썬 코드(`import os; os.system("rm -rf /")` 등)를 그대로 실행할 위험이 있습니다.
* `ast`를 활용하면 소스 코드를 실행하기 전에 **어떤 문법/함수/연산자가 사용되었는지 미리 검사**하여 허용된 구문만 안전하게 실행할 수 있습니다.


2. **코드 정적 분석 (Static Analysis)**
* Flake8, Black, Pylint 같은 정적 분석 도구나 Linter가 코드를 실행하지 않고도 문법 오류, 미사용 변수, 코딩 컨벤션 위반 등을 검사할 때 사용합니다.


3. **코드 변환 및 메타프로그래밍 (Code Transformation)**
* 파이썬 코드를 읽어서 특정 구문을 자동으로 수정하거나 다른 언어로 Transpile(변환)할 수 있습니다.



---

### 2. 핵심 주요 함수

| 함수 | 설명 |
| --- | --- |
| **`ast.parse(source, mode='exec')`** | 소스 코드 문자열을 읽어 AST 트리 노드 객체로 변환합니다.<br>

<br>· `mode='exec'`: 일반 모듈/스크립트 코드 전체<br>

<br>· `mode='eval'`: 단일 수식/표현식 (`3 + 5` 등) |
| **`ast.dump(node, indent=None)`** | AST 트리의 내부 노드 구조를 읽기 쉬운 문자열 형태로 출력해 줍니다. |
| **`ast.literal_eval(node_or_string)`** | 문자열/AST 노드가 파이썬의 기본 리터럴(숫자, 문자열, 튜플, 리스트, 딕셔너리, 불리언)만 포함하고 있는지 안전하게 평가하여 파이썬 객체로 변환합니다. |

---

### 3. 주요 AST 노드 종류 (Node Types)

AST 트리는 문법 요소에 따라 다양한 노드 객체로 구성됩니다.

* **최상위 노드**: `Module` (일반 스크립트), `Expression` (`mode='eval'`일 때의 단일 표현식)
* **표현식 노드 (Expressions)**:
* `ast.Constant`: 상수 값 (숫자 `3`, 문자열 `"hello"`, `True` 등)
* `ast.Name`: 변수 식별자 이름 (`x`, `my_var` 등)
* `ast.BinOp`: 이항 연산 (`3 + 5`, `x * 2` 등)
* `ast.UnaryOp`: 단항 연산 (`-x`, `+5` 등)
* `ast.Call`: 함수 호출 (`print("hello")`)


* **문장 노드 (Statements)**:
* `ast.Assign`: 변수 할당문 (`x = 10`)
* `ast.If`: 조건문 (`if x > 0: ...`)
* `ast.For` / `ast.While`: 반복문
* `ast.FunctionDef`: 함수 정의문 (`def foo(): ...`)
* `ast.Import` / `ast.ImportFrom`: 모듈 임포트문



---



`ast.parse(source, filename='<unknown>', mode='exec')` 함수에서 **`mode` 매개변수**는 입력받은 파이썬 소스 코드 문자열을 **어떤 종류의 문법(Grammar) 구조로 해석하고 검증할지** 결정하는 옵션입니다.

파이썬에는 크게 3가지 `mode`가 존재합니다.

---

### 1. `mode='eval'` (단일 표현식 / 수식)

* **개념**: 단 하나의 평가(Evaluation) 가능한 **표현식(Expression)** 형태만 허용합니다.
* **특징**:
* 값을 반환하는 단일 수식(`3 + 5`), 변수 조회(`x`), 리스트/디렉터리 리터럴(`[1, 2, 3]`), 단일 함수 호출(`sum([1, 2])`) 등만 파싱이 가능합니다.
* 변수 할당문(`x = 10`), 조건문(`if`), 반복문(`for`), 함수/클래스 정의(`def`, `class`), `import` 문과 같이 문장(Statement) 구조가 들어가면 `SyntaxError`가 발생합니다.


* **주요 용도**:
* 계산기, 샌드박스 수식 검증 (`SafePythonSandbox` 구현 시 주로 사용)
* 단일 표현식 정적 검사



```python
import ast

# ✅ 성공: 단일 수식
node = ast.parse("3 + 5 * (10 - 2)", mode='eval')

# ❌ SyntaxError 발생: 변수 할당문(=)은 표현식이 아니라 문장(Statement)임
node = ast.parse("x = 10", mode='eval') 

```

---

### 2. `mode='exec'` (일반 파이썬 모듈 / 기본값)

* **개념**: 일반적인 파이썬 스크립트 파일(`.py`) 전체나 모듈 코드 형태를 파싱하는 기본 옵션입니다.
* **특징**:
* 여러 줄의 코드, 변수 선언, 조건문/반복문, 함수 및 클래스 정의, `import` 문 등 **모든 파이썬 문법 요소**를 제한 없이 해석할 수 있습니다.
* 최상위 노드로 `Module` 객체를 생성합니다.


* **주요 용도**:
* 전체 코드 파일 정적 분석 (Flake8, Pylint 등)
* Linter, Code Formatter(Black 등) 작성



```python
import ast

code = """
import os

def add(a, b):
    return a + b

result = add(3, 5)
"""

# ✅ 성공: 파이썬 모듈 전체 파싱
node = ast.parse(code, mode='exec')

```

---

### 3. `mode='single'` (대화형 REPL 한 줄 입력)

* **개념**: 파이썬 대화형 인터프리터(REPL, CLI 터미널)에 한 줄씩 명령어를 입력할 때의 동작 방식을 시뮬레이션합니다.
* **특징**:
* 단일 문장(Single Statement)을 허용하며, 만약 그 문장이 표현식(Expression)이면 `Interactive` 노드로 감싸져 해당 결과값을 출력할 수 있는 형태로 구성됩니다.


* **주요 용도**:
* 커스텀 파이썬 REPL 쉘/터미널 환경 구현



```python
import ast

# ✅ 성공: 대화형 CLI에 한 줄 입력한 것처럼 동작
node = ast.parse("x = 10", mode='single')
node_expr = ast.parse("3 + 5", mode='single')

```

---

### 📊 `mode` 옵션 요약 비교

| `mode` | 허용하는 문법 구조 | 최상위 노드 (Top Node) | 주요 유스케이스 |
| --- | --- | --- | --- |
| **`eval`** | 단일 표현식만 허용 (`3+5`, `x > 0` 등) | `Expression` | 수식 계산기, 샌드박스 평가 |
| **`exec`** *(기본값)* | 파이썬 코드 전체 (여러 줄, 함수, class, import 등) | `Module` | 정적 분석, 코드 변환 |
| **`single`** | 단일 문장 (REPL 터미널의 1줄 입력 형태) | `Interactive` | 커스텀 REPL CLI 제작 |


### 4. AST 트리를 순회/변환하는 핵심 클래스

`ast` 모듈은 트리를 쉽게 탐색하고 수정할 수 있도록 **방문자 패턴(Visitor Pattern)** 클래스를 제공합니다.

#### ① `ast.NodeVisitor` (순회 및 검사)

트리의 각 노드를 읽기 전용으로 탐색할 때 사용합니다. 노드 타입에 맞춰 `visit_<NodeName>()` 메서드를 정의하면 해당 노드를 만날 때 자동으로 호출됩니다.

In [3]:


import ast

class SecurityChecker(ast.NodeVisitor):
    def visit_Import(self, node):
        print(f"[경고] import 문이 감지되었습니다: {node.names[0].name}")
        self.generic_visit(node)  # 자식 노드 계속 순회

    def visit_Call(self, node):
        # 함수 호출이 식별자(Name) 형태인 경우
        if isinstance(node.func, ast.Name):
            print(f"[기록] 함수 호출 감지: {node.func.id}()")
        self.generic_visit(node)

code = """
import os
print("Hello")
"""
tree = ast.parse(code)
checker = SecurityChecker()
checker.visit(tree)




[경고] import 문이 감지되었습니다: os
[기록] 함수 호출 감지: print()



#### ② `ast.NodeTransformer` (노드 수정 및 변환)

AST 트리의 특정 노드를 다른 노드로 교체하거나 삭제할 때 사용합니다.

- ast.copy_location(new_node, old_node): 특정 단일 노드를 교체할 때 기존 노드의 위치 속성을 새로 생성한 노드에 그대로 이식합니다.

- ast.fix_missing_locations(tree): AST 트리를 변환한 후 compile()을 호출하기 직전에 실행하면, 트리를 전체적으로 순회하며 위치 정보가 누락된 노드를 부모 노드의 위치 정보로 자동 채워주므로 에러를 확실히 방지해 줍니다.



In [5]:
import ast

class DoubleNumberTransformer(ast.NodeTransformer):
    """모든 숫자 상수를 2배로 만드는 변환기"""
    def visit_Constant(self, node):
        if isinstance(node.value, (int, float)):
            # 1. 새로운 노드 생성
            new_node = ast.Constant(value=node.value * 2)
            # 2. 기존 node의 위치 정보(lineno, col_offset 등)를 new_node로 복사
            return ast.copy_location(new_node, node)
        return node

code = "3 + 5"
tree = ast.parse(code, mode='eval')

# 변환 수행
transformer = DoubleNumberTransformer()
transformed_tree = transformer.visit(tree)
print('transformed_tree:', transformed_tree)

# 3. 트리의 누락된 모든 위치 정보를 일괄 수리/보완
ast.fix_missing_locations(transformed_tree)

# 4. 컴파일 및 실행
compiled_code = compile(transformed_tree, filename="<ast>", mode="eval")
print(eval(compiled_code))  # 출력: 16 ((3*2) + (5*2))

transformed_tree: <ast.Expression object at 0x000001E64548ADD0>
16


`3+5` 수식이 파이썬 internal 구문 분석기를 통해 어떤 AST(추상 구문 트리) 구조로 파싱되는지 확인해볼 수 있는 예시 코드입니다.

`ast.dump()` 함수에 `indent=4` 옵션을 전달하면 트리 구조를 한눈에 알아보기 쉽게 정렬하여 출력할 수 있습니다.

---


### 💡 노드 구조 상세 해석

1. **`Expression(...)`**
* 최상위 노드로, 이 파싱 트리가 하나의 평가 가능한 표현식(Expression)임을 의미합니다.


2. **`body=BinOp(...)`**
* 표현식의 주 내용이 이항 연산(Binary Operation)임을 나타냅니다.


3. **`left=Constant(value=3)`**
* 이항 연산자의 좌측 피연산자로 숫자 상수 `3`이 위치함을 나타냅니다.


4. **`op=Add()`**
* 두 피연산자 사이의 연산자가 덧셈(`+`)임을 나타냅니다.


5. **`right=Constant(value=5)`**
* 이항 연산자의 우측 피연산자로 숫자 상수 `5`가 위치함을 나타냅니다.



---

In [6]:
import ast

# 1. 파싱할 수식 문자열 정의
expression = "3+5"

# 2. ast.parse()를 통해 수식을 AST 트키 노드로 변환
# mode='eval'은 단일 표현식(expression) 형태임을 나타냅니다.
parsed_ast = ast.parse(expression, mode='eval')

# 3. ast.dump() 함수를 사용해 AST 트리의 상세 구조 출력
# indent 옵션을 주면 들여쓰기가 적용되어 시각적으로 쉽게 파악할 수 있습니다.
dumped_tree = ast.dump(parsed_ast, indent=4)

print("=== '3+5' 수식의 AST 트리 구조 ===")
print(dumped_tree)

=== '3+5' 수식의 AST 트리 구조 ===
Expression(
    body=BinOp(
        left=Constant(value=3),
        op=Add(),
        right=Constant(value=5)))


(12 + 8) * 3 수식을 ast.dump()로 출력하는 파이썬 코드와 그 결과, 그리고 AST 트리가 괄호와 연산자 우선순위를 어떻게 처리하는지 설명해 드립니다.

`(12 + 8) * 3` 수식을 `ast.dump()`로 출력하는 파이썬 코드와 그 결과, 그리고 AST 트리가 괄호와 연산자 우선순위를 어떻게 처리하는지 설명해 드립니다.



### 💡 괄호와 연산자 우선순위의 반영 원리

파이썬의 AST(추상 구문 트리)에서는  별도의 `Parentheses`(괄호) 노드가 존재하지 않습니다. 대신 괄호와 연산자 우선순위는 트리의 계층 구조(깊이)로 반영됩니다.

1. **최상위 연산자 (`Mult` - 곱셈)**
* 괄호 `(12 + 8)` 전체가 하나의 값으로 계산된 후 `3`과 곱해져야 하므로, 트리 구조의 가장 바깥쪽에 위치한 최상위 연산자는 `Mult`(`*`)입니다.


2. **좌측 자식 노드 (`BinOp` - 덧셈)**
* `Mult` 연산자의 `left`(좌측 피연산자) 자리에 또 다른 `BinOp`(`12 + 8`) 노드가 자식으로 들어가 있습니다.


3. **재귀적 평가 순서**
* 샌드박스의 `_eval_node()`와 같은 재귀 함수가 이 트리를 평가할 때, 최상위 노드(`Mult`)를 계산하기 위해 좌측 자식 노드(`BinOp: 12 + 8`)를 **먼저 실행**하게 됩니다.
* 따라서 괄호 `()`를 통해 지정한 연산 순서가 트리 깊이에 의해 자연스럽게 보장됩니다.



---

In [7]:
import ast

# 1. 괄호가 포함된 복합 수식 파싱
expression = "(12 + 8) * 3"
parsed_ast = ast.parse(expression, mode='eval')

# 2. AST 트리 구조를 들여쓰기하여 출력
print(ast.dump(parsed_ast, indent=4))

Expression(
    body=BinOp(
        left=BinOp(
            left=Constant(value=12),
            op=Add(),
            right=Constant(value=8)),
        op=Mult(),
        right=Constant(value=3)))
